## SHARK-capture:
SHARK-capture (part of the bio-shark suite) is an alignment-free motif detection tool. Unlike SLiMFinder, which looks for exact or wildcard matches, SHARK-capture uses physicochemical similarity. It decomposes your sequences into k-mers and finds regions that "behave" the same way chemically, even if the amino acid letters have mutated.

In [ ]:
!pip install bio-shark==2.0.2

In [ ]:
!pip install openpyxl
import pandas as pd
import os
import re
from bio_shark.capture.compute import run_capture_multiprocessing
from bio_shark.core.utils import read_fasta_file
from pathlib import Path

In [ ]:
if __name__ == "__main__":
  # Load FASTA sequence input file 
  sequence_fasta_input_file: str = "ESX1_IDR_genes.fasta" 
  output_dir: str = "shark_results_esx1" # 
  output_file_name: str = "sharkcapture_consensus_kmers.txt"
  min_kmer_length: int = 3 
  max_kmer_length: int = 10 
  n_output: int = 20 # number of top consensus k-mers to report
  n_processes: int = 4 # number of parallel processes for multi-core processing
  # Comma separated sequence identifiers or substrings to generate output for per-sequence k-mer matches plot, e.g. "sequence_id_1,sequence_id_2". **By default**, plots for all sequences.
  sequence_subset: str = 'all' # 
  # Boolean flag to show scores in log scale (base 10) for per-sequence k-mer matches plot:
  log: bool = False 
  # Boolean flag to enable SHARK-capture Extension Protocol:
  extend: bool = True # 
  #Percentage cutoff for SHARK-capture Extension Protocol, default 0.9:
  cutoff: float = 0.9 
  
  sequence_subset_ls = []
  if sequence_subset == 'all':
    sequence_subset_ls.extend(list(read_fasta_file(file_path=sequence_fasta_input_file).keys()))
  elif sequence_subset:
    sequence_subset_ls.extend(sequence_subset.split(","))
  
  run_capture_multiprocessing(
      sequence_fasta_input_file,
      min_kmer_length,
      max_kmer_length,
      Path(output_dir),
      n_output,
      output_file_name,
      n_processes,
      sequence_subset_ls,
      log,
      extend,
      cutoff
  )



Read fasta file from path ESX1_IDR_genes.fasta; Found 11 sequences; Skipped 0 sequences for having non-canonical AAs
Beginning SHARK-CAPTURE
Read fasta file from path ESX1_IDR_genes.fasta; Found 11 sequences; Skipped 0 sequences for having non-canonical AAs

Processing 3-mers
Collected args (sequence pairs): 66
k=3 - Created master input data file at shark_results_esx1\inputs\k_3.json
Completed processing. Gathered hadamard reciprocals for: 66
Hadamard sorted k-mer score mapping stored at shark_results_esx1\intermediates\hadamards\hadamard_3
Search space (no. unique k-mers): 2254
Hadamard sorted k-mer score mapping stored at shark_results_esx1\intermediates\conserved_kmers\k_3.json

Processing 4-mers
Collected args (sequence pairs): 66
k=4 - Created master input data file at shark_results_esx1\inputs\k_4.json
Completed processing. Gathered hadamard reciprocals for: 66
Hadamard sorted k-mer score mapping stored at shark_results_esx1\intermediates\hadamards\hadamard_4
Search space (no. u

Now to see how many genes these motifs occur in and whether they are in the IDR regions

In [ ]:
# 1. Setup Paths
occ_folder = r"D:\2nd  year\2nd term\Thesis\files from klaas\shark_results_esx1\outputs\occurrences"
idr_excel_path = r"D:\2nd  year\2nd term\Thesis\files from klaas\IDR_positions.xlsx"

# 2. Load IDR Data and select only relevant columns
idr_df = pd.read_excel(idr_excel_path, usecols=['protein', 'position IDR'])
# Fill merged cells by copying the protein name down to empty rows
idr_df['protein'] = idr_df['protein'].ffill()
# Clean up column names in case of trailing spaces
idr_df.columns = [c.strip() for c in idr_df.columns]
# Create a lowercase version of the protein column for easy matching
idr_df['protein_lower'] = idr_df['protein'].astype(str).str.lower()

def is_in_idr(protein_name, start, end):
    """Checks if motif coordinates overlap with any IDR range for that protein."""
    p_name_lower = str(protein_name).lower()
    # Correct for naming discordance between the IDR file and shark-capture results
    if p_name_lower == 'rv3863':
        p_name_lower = 'espm'
    # Get all rows matching this protein
    matching_rows = idr_df[idr_df['protein_lower'] == p_name_lower]
    if matching_rows.empty:
        return "Protein Not Found"
    
    # Loop through every row found for this protein
    for _, row in matching_rows.iterrows():
        idr_ranges_str = str(row['position IDR'])
    
        ranges = re.findall(r'(\d+)-(\d+)', idr_ranges_str)
    
        for r_start, r_end in ranges:
            # Check if the motif is fully contained within this specific IDR segment
            if start >= int(r_start) and end <= int(r_end):
                return "Yes"
    return "No"

# 3. Process Shark-Capture Occurrences
results = []

for file in os.listdir(occ_folder):
    # Only process files that start with 'sharkcapture_' and ignore the 'all_occurrences' summaries
    if file.startswith("sharkcapture_") and file.endswith(".tsv"):
        file_path = os.path.join(occ_folder, file)
        try:
            # Load the TSV file
            df = pd.read_csv(file_path, sep='\t')
            
            for _, row in df.iterrows():
                # Extract gene name: "Mycobacterium tuberculosis H37Rv|Rv3614c|espD" -> "espD"
                full_id = str(row['sequenceID'])
                protein_name = full_id.split('|')[-1] 
                
                results.append({
                    'motif': row['reference_kmer'], 
                    'protein_original': protein_name,
                    'protein_lower': protein_name.lower(),
                    'start': int(row['start']),
                    'end': int(row['end'])
                })
        except Exception as e:
            print(f"Skipping {file} due to error: {e}")

# 4. Aggregate and Label
if not results:
    print("No results found. Check your file paths and column names.")
else:
    final_df = pd.DataFrame(results)

# Count unique genes per motif
motif_summary = final_df.groupby('motif')['protein_lower'].nunique().reset_index()
motif_summary.columns = ['motif', 'Gene_Count']

# 5. Detailed view with IDR check
final_df['In_IDR'] = final_df.apply(lambda x: is_in_idr(x['protein_lower'], x['start'], x['end']), axis=1)

# Merge Gene Count back into the detailed view
final_df = final_df.merge(motif_summary, on='motif')

# Reorder columns for the final report
final_output = final_df[['motif', 'protein_original', 'start', 'end', 'Gene_Count', 'In_IDR']]

# 6. Save and Preview
print(final_output.sort_values(by='Gene_Count', ascending=False).head(20))
final_output.to_csv("Motif_IDR_Stability_Report.csv", index=False)



Defaulting to user installation because normal site-packages is not writeable
    motif protein_original  start  end  Gene_Count In_IDR
138   ALD             espK    496  498          11     No
117   AAT             espE    222  224          11    Yes
190   EEA             espH    157  159          11     No
189   EEA             espE    382  384          11    Yes
112   AAT             lsr2    110  112          11     No
113   AAT             espD     38   40          11    Yes
114   AAT             espA    299  301          11     No
115   AAT           Rv3863    274  276          11     No
116   AAT           Rv3863    235  237          11    Yes
118   AAT             espH     21   23          11    Yes
127   ALD             lsr2     74   76          11    Yes
119   AAT            PPE68     67   69          11     No
120   AAT            PPE68    204  206          11     No
121   AAT             espI    328  330          11    Yes
122   AAT             espJ    197  199          11  

In [ ]:
# 7. Check the biological function of the motifs found in IDRs
idr_motifs = final_df[final_df['In_IDR'] == "Yes"]
unique_kmers = sorted(idr_motifs['motif'].unique())
    
# Create the ELM Query Report
elm_data = []
for kmer in unique_kmers:
    elm_data.append({
        'Motif_Sequence': kmer,
        'ELM_Instance_Search': f"http://elm.eu.org/instances.html?q={kmer}",
        'Found_In_Proteins': ", ".join(idr_motifs[idr_motifs['motif'] == kmer]['protein_lower'].unique())
    })
    
elm_df = pd.DataFrame(elm_data)
elm_df.to_csv("ELM_Search_Queries.csv", index=False)
    
print(f"Success! {len(unique_kmers)} unique IDR motifs exported.")


Success! 20 unique IDR motifs exported.
Example Query: http://elm.eu.org/instances.html?q=AAA
